[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Pipelines


## What you will be able to do

Join a scaler and a model into one pipeline that is fitted, scored and asked for answers as a single
object, and see, on columns of pure noise, how choosing columns before cross-validation inflates the
score.

## The idea

### The problem

A logistic regression that predicts snow on 1 May works better on scaled features, so the
**Preprocessing** notebook fitted a `StandardScaler` on the training stations, transformed them,
fitted the model on the result, and transformed the test stations with the same scaler before
scoring. That is three objects to keep in step, and every new station needs the same two calls in
the same order before the model sees it. Leave out the transform once and the model receives raw
elevations and years where it learned standardized ones, and it answers without an error.

Cross-validation makes it worse. Each fold should fit every step on its own training rows, and a
scaler or a column selector fitted once on the whole table has already seen the rows each fold tests
on. A scaler learns only each column's mean and spread, and fitted that way it barely moves the
score. With many columns and a selector that keeps the ones that look best against the answers, it
is large, because the selector finds columns that happen to fit the test rows, and the score
measures that luck.

### What a pipeline is

> A **pipeline** is an estimator made of steps that run in order. Every step but the last is a
> transformer, and the last can be any estimator. **`fit`** fits each step in turn on the output of
> the one before it, and **`predict`** and **`score`** send new rows through the fitted
> transformers before the last step answers.

### Why it works that way

A pipeline is a single estimator, so anything that takes an estimator takes a pipeline.
`cross_val_score` fits the whole chain again on each fold's training rows, so the scaler and the
selector learn nothing from the test rows, and the score is of the chain as it would run on stations
nobody has measured. One `predict` call does every step, so the transform cannot be forgotten.

Every step has a name. `make_pipeline` names a step after its class in lower case, such as
`standardscaler`, and `Pipeline` takes the names you give it. The names are how you reach a step's
settings from outside, as the step's name, two underscores and the setting, such as
`logisticregression__C`, which the **Hyperparameter Search** notebook relies on. After `fit`, each
step's learned attributes are still there, through `named_steps` or an index.

### Where this shows up

A pipeline belongs in almost any model that needs its data prepared. The **Cross-Validation**
notebook scores pipelines, the **Hyperparameter Search** notebook searches their settings, the
**Mixed Column Types** notebook puts a whole table's preparation into one step, and the **Saving a
Model** notebook saves a fitted pipeline as one file, with its preparation inside it.

### What this notebook covers

- a scaler and a logistic regression as one pipeline, fitted and scored once
- the fitted steps inside it, and new rows sent through every step
- named steps, and settings reached as `step__setting`
- the leak a pipeline prevents, run on columns of pure noise
- three errors, one of them a model given raw rows that answers anyway

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

model = make_pipeline(StandardScaler(), LinearRegression())
model.fit([[1.0, 100.0], [2.0, 300.0], [3.0, 200.0], [4.0, 400.0]], [10.0, 21.0, 29.0, 41.0])
print(model.predict([[5.0, 500.0]]).round(1))
print(list(model.named_steps))
```

```
[51.1]
['standardscaler', 'linearregression']
```

`fit` scaled the four rows and fitted the linear model on the scaled values. `predict` scaled the
new row with the means and spreads it had learned, then used the model. The two steps are named
after their classes.

## Setup

Two libraries for the data, scikit-learn, and the tools a pipeline is built from.

- `numpy` and `pandas` build the stations and the columns of noise, and the cell prints
  scikit-learn's version
- `make_pipeline` and `Pipeline` build pipelines, the first naming the steps for you
- `StandardScaler` and `LogisticRegression` are the two steps of the snow model
- `train_test_split` holds a quarter of the stations back, and `cross_val_score` scores by folds
- `SelectKBest` and `f_classif` keep the columns that look most related to the answer, in the
  demonstration of the leak

`station_network` makes a network of 1,200 weather stations in Norway. The stations are made up, and
a generator started from a fixed seed draws them, so the table is the same on every run. Each row
has a station's latitude, longitude, elevation in meters, distance from the coast in kilometers and
the year it was installed, its terrain and owner as text, and three things to predict, of which this
notebook uses one: whether the station has snow on 1 May. `FEATURES` names the five numeric columns
the models use.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


## Worked examples

### A scaler and a model as one estimator

The stations are split as in the **Train and Test** notebook, with a quarter held back and the share
of snow kept the same on both sides:

In [2]:
X = network[FEATURES]
snow = network["snow_on_may_1"]
X_train, X_test, y_train, y_test = train_test_split(
    X, snow, test_size=0.25, random_state=0, stratify=snow
)

pipeline = make_pipeline(StandardScaler(), LogisticRegression())
pipeline

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression())])

The pipeline shows its two steps, with the names `make_pipeline` gave them. It has not learned
anything yet. A single `fit` and a single `score` do the whole chain:

In [3]:
pipeline.fit(X_train, y_train)

print("accuracy on the held-back stations:", round(pipeline.score(X_test, y_test), 3))

accuracy on the held-back stations: 0.84


`fit` fitted the scaler on the 900 training stations, transformed them, and fitted the logistic
regression on the result. `score` transformed the 300 held-back stations with that same scaler
before the model predicted them, and 84 percent of its answers were right.

### The fitted steps inside it

`named_steps` reaches each step by name, and an index reaches it by position, so `pipeline[-1]` is
the model:

In [4]:
print(list(pipeline.named_steps))
print("means:", pipeline.named_steps["standardscaler"].mean_.round(2))
print("coefficients:", pipeline[-1].coef_.round(2))

['standardscaler', 'logisticregression']
means: [  64.52   17.39  290.69   24.59 1987.76]
coefficients: [[ 1.78  0.65  1.2   1.14 -0.09]]


The scaler's means are those of the training stations alone, since only they reached `fit`. Because
the model learned from standardized columns, its coefficients are comparable with each other, and a
step of one standard deviation in latitude moves the model's odds of snow the most, and one in
`installed` the least.

### New rows go through every step

`predict` takes stations as they are in the table:

In [5]:
new_rows = X_test.head(3)
by_pipeline = pipeline.predict(new_rows)

scaler = pipeline.named_steps["standardscaler"]
model = pipeline.named_steps["logisticregression"]
by_hand = model.predict(scaler.transform(new_rows))

print(by_pipeline, by_hand)
print("the same:", np.array_equal(by_pipeline, by_hand))

[False False False] [False False False]
the same: True


Done by hand, the same answers need the scaler's `transform` and then the model's `predict`. The
pipeline makes both calls, in that order, every time.

### Naming the steps yourself

`Pipeline` takes a list of pairs, each a name and a step. Shorter names make the settings easier to
reach:

In [6]:
named = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression())])
named.set_params(model__C=0.1)

print(named)
print(named.get_params()["model__C"])

Pipeline(steps=[('scale', StandardScaler()),
                ('model', LogisticRegression(C=0.1))])
0.1


`model__C` is the setting `C` of the step called `model`, the two joined by two underscores. `C`
controls how strongly a logistic regression is held back from fitting the training rows closely,
and smaller values hold it back more. The pipeline passes the setting to the step, and `named` is
otherwise the same pipeline as before.

### The leak a pipeline prevents

The leak is easiest to see where there is nothing real to find. Here are 200 rows of 2,000 columns
of random numbers, and an answer for each row drawn like a coin flip, with nothing connecting the
two:

In [7]:
noise = pd.DataFrame(np.random.default_rng(0).normal(size=(200, 2000)))
coin = np.random.default_rng(1).integers(0, 2, 200)

print(noise.shape, "| share of ones:", coin.mean().round(2))

(200, 2000) | share of ones: 0.52


A model can do no better than guessing on these rows, which is right half the time. The first way
below chooses the 20 columns most related to the answer using every row, and then cross-validates a
model on those 20. The second puts the same choice inside a pipeline, so each fold chooses its 20
from its own training rows:

In [8]:
chosen_once = SelectKBest(f_classif, k=20).fit(noise, coin)
outside = cross_val_score(LogisticRegression(), chosen_once.transform(noise), coin, cv=5)

inside = cross_val_score(
    make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression()), noise, coin, cv=5
)

print("chosen outside the folds:", outside.mean().round(3))
print("chosen inside each fold: ", inside.mean().round(3))

chosen outside the folds: 0.78
chosen inside each fold:  0.5


Chosen outside the folds, the columns scored 0.78 on data with nothing in it. Among 2,000 random
columns, some happen to line up with the coin flips, and choosing them with every row included the
rows each fold later tested on. The pipeline chose each fold's columns without its test rows, and
scored 0.5, the coin flip it really is. The selector chose by looking at the answers, which is what
made the leak this large. The same thing happens with real data whenever a step that learns from the
answers is fitted on rows it is then scored on, and there it is harder to see, because nobody knows
what the honest score should be.

### Cross-validating the whole chain

The station pipeline cross-validates the same way. Each of the five folds refits the scaler and the
model on its own training rows:

In [9]:
scores = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), X, snow, cv=5)

print(scores.round(3), "mean", scores.mean().round(3))

[0.808 0.854 0.858 0.854 0.838] mean 0.842


The five scores are each from a scaler and a model that never saw that fold's stations. The
**Cross-Validation** notebook looks at how the folds are made and what their spread says. Here is
the same model with the scaler fitted once on every station, outside the folds:

In [10]:
scaled_once = StandardScaler().fit_transform(X)
outside_scaler = cross_val_score(LogisticRegression(), scaled_once, snow, cv=5)

print(outside_scaler.round(3), "mean", outside_scaler.mean().round(3))

[0.808 0.854 0.858 0.854 0.838] mean 0.842


Fitted once on all 1,200 stations, the scaler gives the same five scores to three places. It learns
only each column's mean and spread and never sees the answers, so the test rows told it almost
nothing. The pipeline keeps it inside the folds anyway, so that the same code stays correct when a
step that learns from the answers joins it.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/05-pipelines-solutions.ipynb).

**1.** Make a pipeline of `StandardScaler` and `LogisticRegression` that predicts `snow_on_may_1` from `latitude` and `elevation_m` alone, fit it on a split with `test_size=0.25`, `random_state=0` and `stratify`, and print its accuracy on the held-back stations.

In [11]:
# your code here

**2.** For the pipeline from task 1, print the names of its steps and the means its scaler learned.

In [12]:
# your code here

**3.** Change the pipeline's `C` to 0.01 with `set_params`, fit it again, and print its accuracy on the held-back stations.

In [13]:
# your code here

**4.** Cross-validate a pipeline of `StandardScaler` and `LogisticRegression` on all five `FEATURES` and `snow_on_may_1` with `cv=10`, and print the ten scores and their mean, rounded to three places.

In [14]:
# your code here

**5.** Repeat the noise demonstration with `k=5` columns instead of 20, both outside and inside the folds, and print the two mean scores.

In [15]:
# your code here

## Common errors

### ValueError: Invalid parameter 'C' for estimator Pipeline(steps=[('standardscaler', StandardScaler()), ('logisticregression', LogisticRegression())]).

In [16]:
make_pipeline(StandardScaler(), LogisticRegression()).set_params(C=0.1)

ValueError: Invalid parameter 'C' for estimator Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression())]). Valid parameters are: ['memory', 'steps', 'transform_input', 'verbose'].

A pipeline's own settings are the four the message lists, and `C` belongs to one of its steps. Name
the step first, `logisticregression__C=0.1`, which is the step's name, two underscores and the
setting.

### TypeError: All intermediate steps should be transformers and implement fit and transform or be the string 'passthrough' 'LogisticRegression()' (type <class 'sklearn.linear_model._logistic.LogisticRegression'>) doesn't

In [17]:
make_pipeline(LogisticRegression(), StandardScaler()).fit(X_train, y_train)

TypeError: All intermediate steps should be transformers and implement fit and transform or be the string 'passthrough' 'LogisticRegression()' (type <class 'sklearn.linear_model._logistic.LogisticRegression'>) doesn't

The steps run in the order they are listed, and every step but the last has to pass the rows on with
`transform`. A logistic regression has no `transform`, so it can only come last. Swap the two.

### No error, and 290 of 300 stations predicted to have snow

In [18]:
scaler = StandardScaler().set_output(transform="pandas").fit(X_train)
model = LogisticRegression().fit(scaler.transform(X_train), y_train)

print("predicted snow:", model.predict(X_test).sum(), "of", len(X_test))
print("measured snow: ", y_test.sum())
print("accuracy given raw rows:   ", round(model.score(X_test, y_test), 3))
print("accuracy given scaled rows:", round(model.score(scaler.transform(X_test), y_test), 3))

predicted snow: 290 of 300
measured snow:  116
accuracy given raw rows:    0.42
accuracy given scaled rows: 0.84


Without a pipeline, the model learned from standardized columns under the table's own column names,
and was then given the stations as they are, with elevations in the hundreds and years near 2000
where it expected values near zero. It answered every row anyway, and predicted snow at 290 stations
where 116 had it. Nothing raised, and nothing warned, because the columns had the names the model
learned, and to the model they were simply rows of five numbers. `set_output(transform="pandas")`
made the scaler return a DataFrame with those names, as code that keeps its tables in pandas often
does. A pipeline makes the scaler part of the model, so the only way to predict is through it.

## Recap

- Join transformers and a model with `make_pipeline` or `Pipeline`, and fit, score and predict with
  it as one estimator
- Reach a fitted step with `named_steps` or an index, and read what it learned
- Change a step's setting from outside as the step's name, two underscores and the setting
- Explain why a step that learns from the answers on every row inflates a cross-validated score, and
  keep every step inside the pipeline

## What is next

The **Cross-Validation** notebook looks at the folds themselves: how they are made, why they are
shuffled, and what the spread of their scores tells you.

---

&#8592; **Previous:** [Preprocessing](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/04-preprocessing.ipynb)  &nbsp;·&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Cross-Validation](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/06-cross-validation.ipynb) &#8594;
